# 📦 Notebook 1: Estructuras de Datos Nativas de Python para ML
### Curso: Machine Learning Avanzado — Módulo 1
**Objetivo:** Dominar listas, diccionarios, conjuntos y tuplas con enfoque en rendimiento y aplicaciones reales en ciencia de datos.

---
## 🗺️ Contenido
1. Listas: operaciones avanzadas y rendimiento
2. List Comprehensions vs loops (benchmark)
3. Diccionarios: estructuras clave en ML pipelines
4. Sets y Tuplas: cuándo y por qué usarlos
5. Estructuras anidadas: representación de datos reales
6. Caso aplicado: Pipeline de preprocesamiento con estructuras nativas

---
## 1. Listas en Python: Más Allá de lo Básico

### 📖 Concepto
Una **lista** es una colección ordenada, mutable y heterogénea. En ML, las usamos para almacenar:
- Secuencias de predicciones
- Resultados de validación cruzada
- Pipelines de transformaciones
- Historiales de métricas durante el entrenamiento

**Complejidad computacional clave:**
| Operación | Complejidad |
|-----------|-------------|
| `append()` | O(1) amortizado |
| `insert(i, x)` | O(n) |
| `x in list` | O(n) |
| `list[i]` | O(1) |

In [ ]:
import time
import sys
from collections import defaultdict, OrderedDict

# ── Operaciones avanzadas con listas ──────────────────────────────────────

# Simulación: historial de loss durante entrenamiento de una red neuronal
training_loss = [0.95, 0.87, 0.76, 0.65, 0.54, 0.48, 0.43, 0.39, 0.37, 0.35]
val_loss      = [0.98, 0.91, 0.82, 0.74, 0.68, 0.65, 0.63, 0.64, 0.66, 0.69]

# Slicing avanzado: detectar overfitting (val_loss empieza a subir)
best_epoch = val_loss.index(min(val_loss))
print(f"Mejor época: {best_epoch + 1} | Val Loss mínimo: {val_loss[best_epoch]:.4f}")

# Zip: análisis de gap train/val por época
gaps = [abs(v - t) for t, v in zip(training_loss, val_loss)]
print(f"\nGap train/val por época: {[round(g, 4) for g in gaps]}")
print(f"Época con mayor overfitting: {gaps.index(max(gaps)) + 1}")

# Enumerate con condición: épocas con mejora significativa (>5%)
mejoras = [
    (i + 1, round((val_loss[i-1] - val_loss[i]) / val_loss[i-1] * 100, 2))
    for i in range(1, len(val_loss))
    if (val_loss[i-1] - val_loss[i]) / val_loss[i-1] > 0.05
]
print(f"\nÉpocas con mejora >5%: {mejoras}")

In [ ]:
# ── Benchmark: Loop vs List Comprehension vs map() ────────────────────────
import timeit

# Tarea: normalizar 100,000 valores entre 0 y 1 (Min-Max Scaling)
import random
data = [random.uniform(0, 1000) for _ in range(100_000)]
min_val, max_val = min(data), max(data)

# Método 1: Loop tradicional
def normalize_loop(data, mn, mx):
    result = []
    for x in data:
        result.append((x - mn) / (mx - mn))
    return result

# Método 2: List Comprehension
def normalize_lc(data, mn, mx):
    return [(x - mn) / (mx - mn) for x in data]

# Método 3: map() con lambda
def normalize_map(data, mn, mx):
    return list(map(lambda x: (x - mn) / (mx - mn), data))

t1 = timeit.timeit(lambda: normalize_loop(data, min_val, max_val), number=100)
t2 = timeit.timeit(lambda: normalize_lc(data, min_val, max_val), number=100)
t3 = timeit.timeit(lambda: normalize_map(data, min_val, max_val), number=100)

print("Benchmark (100 iteraciones sobre 100K elementos):")
print(f"  Loop tradicional  : {t1:.4f}s")
print(f"  List Comprehension: {t2:.4f}s  ← Recomendado")
print(f"  map() + lambda    : {t3:.4f}s")
print(f"\nList Comprehension es {t1/t2:.2f}x más rápido que el loop")

---
## 2. Diccionarios: El Corazón de los Pipelines en ML

### 📖 Concepto
Los **diccionarios** en Python 3.7+ mantienen orden de inserción y tienen complejidad O(1) para búsqueda, inserción y eliminación. Son fundamentales en ML para:
- Representar hiperparámetros
- Almacenar métricas por modelo o experimento (como MLflow)
- Encodings de categorías (Label Encoding manual)
- Feature stores ligeros

**Tipos especializados del módulo `collections`:**
- `defaultdict`: evita `KeyError`, ideal para agrupar datos
- `Counter`: frecuencias de categorías
- `OrderedDict`: historial ordenado de experimentos

In [ ]:
from collections import defaultdict, Counter
import json

# ── Caso Real: Registro de experimentos de ML ─────────────────────────────
# Estructura de un experimento (similar a MLflow runs)
experimento = {
    "run_id": "exp_001",
    "modelo": "RandomForest",
    "hiperparametros": {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_split": 5,
        "class_weight": "balanced"
    },
    "metricas": {
        "train": {"accuracy": 0.94, "f1_macro": 0.93, "roc_auc": 0.97},
        "val":   {"accuracy": 0.89, "f1_macro": 0.88, "roc_auc": 0.93}
    },
    "features_importancia": {
        "edad": 0.32, "ingreso_mensual": 0.28,
        "historial_credito": 0.21, "deuda_actual": 0.12, "otros": 0.07
    }
}

# Acceso seguro con .get() (nunca KeyError)
print("Accuracy de validación:", experimento["metricas"]["val"].get("accuracy", "N/A"))

# Feature más importante
top_feature = max(experimento["features_importancia"],
                  key=experimento["features_importancia"].get)
print(f"Feature más importante: {top_feature} "
      f"({experimento['features_importancia'][top_feature]:.0%})")

# Detectar overfitting automáticamente
gap_acc = experimento["metricas"]["train"]["accuracy"] - \
          experimento["metricas"]["val"]["accuracy"]
print(f"Gap Train-Val Accuracy: {gap_acc:.4f} ",
      "⚠️ Posible overfitting" if gap_acc > 0.05 else "✅ OK")

In [ ]:
# ── defaultdict: Agrupar predicciones por clase ───────────────────────────
# Caso: análisis de distribución de predicciones por clase
y_pred = ["spam", "ham", "spam", "spam", "ham", "phishing",
           "ham", "spam", "phishing", "ham", "spam", "ham"]
y_true = ["spam", "ham", "ham",  "spam", "ham", "phishing",
           "ham", "spam", "ham",  "ham", "spam", "spam"]

# Agrupar errores por clase verdadera
errores_por_clase = defaultdict(list)
for i, (pred, true) in enumerate(zip(y_pred, y_true)):
    if pred != true:
        errores_por_clase[true].append({"idx": i, "predicho": pred})

print("Errores de clasificación por clase real:")
for clase, errores in errores_por_clase.items():
    print(f"  Clase '{clase}': {len(errores)} errores → {errores}")

# Counter: distribución de clases (detección de desbalanceo)
distribucion = Counter(y_true)
total = len(y_true)
print("\nDistribución de clases (y_true):")
for clase, count in distribucion.most_common():
    print(f"  {clase:10s}: {count:2d} muestras ({count/total:.0%})")

mayoria = distribucion.most_common(1)[0]
minoria = distribucion.most_common()[-1]
ratio = mayoria[1] / minoria[1]
print(f"\nRatio desbalanceo: {ratio:.1f}:1",
      "⚠️ Aplicar SMOTE/class_weight" if ratio > 3 else "✅ Balanceado")

---
## 3. Tuplas y Sets: Estructuras Subestimadas en DS

### 📖 Concepto

**Tuplas** — inmutables, hashables:
- Representar coordenadas, pares (feature, label), configuraciones fijas
- Usar como **claves de diccionarios** (los dicts no aceptan listas como claves)
- Named tuples: registros ligeros sin overhead de clases

**Sets** — sin duplicados, O(1) para `in`:
- Vocabulario de NLP
- Detección rápida de categorías vistas/no vistas
- Validación de features entre train y test

In [ ]:
from collections import namedtuple

# ── Named Tuples: Registros de predicciones ───────────────────────────────
Prediccion = namedtuple("Prediccion", ["id_muestra", "clase", "probabilidad", "modelo"])

predicciones = [
    Prediccion(id_muestra=101, clase="fraude",    probabilidad=0.94, modelo="XGBoost"),
    Prediccion(id_muestra=102, clase="legítimo",  probabilidad=0.87, modelo="XGBoost"),
    Prediccion(id_muestra=103, clase="fraude",    probabilidad=0.78, modelo="LightGBM"),
    Prediccion(id_muestra=104, clase="legítimo",  probabilidad=0.92, modelo="XGBoost"),
]

# Filtrar predicciones de alta confianza de fraude
alertas = [p for p in predicciones if p.clase == "fraude" and p.probabilidad > 0.80]
print("Alertas de fraude (confianza > 80%):")
for alerta in alertas:
    print(f"  Muestra {alerta.id_muestra}: {alerta.probabilidad:.0%} [{alerta.modelo}]")

# ── Sets: Validación de features train vs test ────────────────────────────
features_train = {"edad", "ingreso", "historial_credito", "deuda", "n_productos", "region"}
features_test  = {"edad", "ingreso", "historial_credito", "deuda", "n_productos", "saldo"}

features_nuevas   = features_test - features_train   # en test pero NO en train
features_faltantes = features_train - features_test  # en train pero NO en test
features_comunes  = features_train & features_test

print("\nValidación de features Train vs Test:")
print(f"  ✅ Comunes        : {features_comunes}")
print(f"  ⚠️ Solo en test   : {features_nuevas} ← Data leakage riesgo")
print(f"  ❌ Faltantes test : {features_faltantes} ← Modelo fallará en producción")

---
## 4. Caso Integrador: Pipeline de Preprocesamiento con Estructuras Nativas

**Industria: Fintech — Scoring Crediticio**

Construiremos un mini-pipeline de preprocesamiento usando **exclusivamente estructuras nativas de Python** antes de introducir Pandas/NumPy, para entender la base computacional.

In [ ]:
import statistics
from functools import reduce

# Dataset crudo: clientes banco (lista de dicts — formato JSON-like)
clientes = [
    {"id": 1, "edad": 35, "ingreso": 45000, "deuda": 12000, "historial": "bueno",  "mora": False},
    {"id": 2, "edad": 28, "ingreso": None,  "deuda": 8000,  "historial": "regular","mora": True},
    {"id": 3, "edad": 52, "ingreso": 90000, "deuda": 5000,  "historial": "bueno",  "mora": False},
    {"id": 4, "edad": 19, "ingreso": 18000, "deuda": 0,     "historial": "sin_hist","mora": False},
    {"id": 5, "edad": 45, "ingreso": 65000, "deuda": 25000, "historial": "malo",   "mora": True},
    {"id": 6, "edad": 38, "ingreso": 52000, "deuda": 15000, "historial": "bueno",  "mora": False},
]

# PASO 1: Imputación de valores faltantes (mediana)
ingresos_validos = [c["ingreso"] for c in clientes if c["ingreso"] is not None]
mediana_ingreso = statistics.median(ingresos_validos)

clientes_limpios = [
    {**c, "ingreso": c["ingreso"] if c["ingreso"] is not None else mediana_ingreso}
    for c in clientes
]

# PASO 2: Feature Engineering — ratio deuda/ingreso
for c in clientes_limpios:
    c["ratio_deuda"] = round(c["deuda"] / c["ingreso"], 4) if c["ingreso"] > 0 else 0

# PASO 3: Encoding ordinal de historial_credito
encoding_historial = {"malo": 0, "sin_hist": 1, "regular": 2, "bueno": 3}
for c in clientes_limpios:
    c["historial_enc"] = encoding_historial.get(c["historial"], -1)

# PASO 4: Normalización Min-Max manual
def min_max_normalize(data, key):
    vals = [d[key] for d in data]
    mn, mx = min(vals), max(vals)
    for d in data:
        d[f"{key}_norm"] = round((d[key] - mn) / (mx - mn), 4) if mx != mn else 0

for feature in ["edad", "ingreso", "ratio_deuda", "historial_enc"]:
    min_max_normalize(clientes_limpios, feature)

# PASO 5: Selección de features para el modelo
features_modelo = ["edad_norm", "ingreso_norm", "ratio_deuda_norm", "historial_enc_norm"]
X = [[c[f] for f in features_modelo] for c in clientes_limpios]
y = [int(c["mora"]) for c in clientes_limpios]

print("Dataset preprocesado (X):")
print(f"{'ID':4} | {' | '.join([f[:8]:^10] for f in features_modelo)} | mora")
print("-" * 70)
for i, (row, label) in enumerate(zip(X, y)):
    vals = " | ".join(f"{v:10.4f}" for v in row)
    print(f"{clientes_limpios[i]['id']:4} | {vals} | {'✓ Mora' if label else '✗ OK'}")

---
## 🧠 Reflexión Final del Notebook 1

### Lo que aprendimos:
| Estructura | Cuándo usarla en ML |
|------------|--------------------|
| **Lista** | Secuencias de métricas, pipelines, batches de datos |
| **Dict** | Hiperparámetros, experimentos, feature stores, encodings |
| **Tuple** | Configuraciones fijas, claves compuestas, named records |
| **Set** | Vocabularios NLP, validación train/test, deduplicación |
| **defaultdict** | Agrupación de errores, acumuladores |
| **Counter** | Distribución de clases, análisis de desbalanceo |
| **namedtuple** | Registros ligeros de predicciones, sin overhead de clase |

### ⚡ Reglas de Oro:
1. Siempre preferir **list comprehension** sobre loops para transformaciones
2. Usar `.get()` en dicts para evitar `KeyError` en datos de producción
3. Los **sets** son O(1) para membership testing — úsalos para validaciones
4. **NumPy** reemplaza listas para operaciones numéricas masivas (siguiente notebook)